# Experiment 19: Feature Engineering + XGBoost

Test progressively stronger feature representations using the same validation split and XGBoost baseline.

Previous local best: **0.941815**

Goal: determine whether engineered behavioral, ratio, interaction, and log features unlock signal that the raw-feature models are missing.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

PROJECT_ROOT = Path(r'C:\Users\aakif\Documents\DataCompetition')
DATA_PATH = PROJECT_ROOT / 'data' / 'train.csv'

train = pd.read_csv(DATA_PATH)
X = train.drop(columns=['Will_Buy_EV', 'id']).copy()
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train shape:', X_train.shape)
print('Validation shape:', X_valid.shape)
print('Positive rate:', y.mean())

Train shape: (534932, 13)
Validation shape: (133733, 13)
Positive rate: 0.17464500160768098


In [2]:
def add_engineered_features(df, level):
    df = df.copy()

    def safe_ratio(a, b):
        return a / (b.replace(0, np.nan) + 1e-6)

    page_cols = [
        c for c in ['Administrative', 'Informational', 'ProductRelated']
        if c in df.columns
    ]
    duration_cols = [
        c for c in [
            'Administrative_Duration',
            'Informational_Duration',
            'ProductRelated_Duration'
        ] if c in df.columns
    ]

    if level >= 1:
        if page_cols:
            df['Total_PageViews'] = df[page_cols].sum(axis=1)
        if duration_cols:
            df['Total_Duration'] = df[duration_cols].sum(axis=1)

        if 'ProductRelated' in df.columns and 'Total_PageViews' in df.columns:
            df['Product_Page_Share'] = safe_ratio(df['ProductRelated'], df['Total_PageViews'])

        if 'ProductRelated_Duration' in df.columns and 'Total_Duration' in df.columns:
            df['Product_Duration_Share'] = safe_ratio(
                df['ProductRelated_Duration'],
                df['Total_Duration']
            )

        if 'Total_PageViews' in df.columns and 'Total_Duration' in df.columns:
            df['Duration_Per_Page'] = safe_ratio(
                df['Total_Duration'],
                df['Total_PageViews']
            )

        if 'ProductRelated' in df.columns and 'ProductRelated_Duration' in df.columns:
            df['Product_Duration_Per_Page'] = safe_ratio(
                df['ProductRelated_Duration'],
                df['ProductRelated']
            )

        if 'PageValues' in df.columns and 'Total_PageViews' in df.columns:
            df['PageValue_Per_Page'] = safe_ratio(
                df['PageValues'],
                df['Total_PageViews']
            )

    if level >= 2:
        if 'ProductRelated' in df.columns and 'PageValues' in df.columns:
            df['Product_PageValue_Interaction'] = (
                df['ProductRelated'] * df['PageValues']
            )

        if 'ProductRelated_Duration' in df.columns and 'PageValues' in df.columns:
            df['Product_Duration_PageValue_Interaction'] = (
                df['ProductRelated_Duration'] * df['PageValues']
            )

        if 'BounceRates' in df.columns and 'ExitRates' in df.columns:
            df['Bounce_Exit_Gap'] = df['ExitRates'] - df['BounceRates']
            df['Bounce_Exit_Ratio'] = safe_ratio(df['BounceRates'], df['ExitRates'])

        if 'Weekend' in df.columns and 'ProductRelated' in df.columns:
            df['Weekend_Product_Interaction'] = (
                df['Weekend'].astype(int) * df['ProductRelated']
            )

        if 'Weekend' in df.columns and 'PageValues' in df.columns:
            df['Weekend_PageValue_Interaction'] = (
                df['Weekend'].astype(int) * df['PageValues']
            )

    if level >= 3:
        log_candidates = [
            'Administrative',
            'Administrative_Duration',
            'Informational',
            'Informational_Duration',
            'ProductRelated',
            'ProductRelated_Duration',
            'PageValues',
            'Total_PageViews',
            'Total_Duration'
        ]
        for col in log_candidates:
            if col in df.columns:
                df[f'Log_{col}'] = np.log1p(df[col].clip(lower=0))

    if level >= 4:
        if 'Month' in df.columns and 'VisitorType' in df.columns:
            df['Month_VisitorType'] = (
                df['Month'].astype(str) + '_' + df['VisitorType'].astype(str)
            )

        if 'TrafficType' in df.columns and 'VisitorType' in df.columns:
            df['Traffic_VisitorType'] = (
                df['TrafficType'].astype(str) + '_' + df['VisitorType'].astype(str)
            )

        if 'OperatingSystems' in df.columns and 'Browser' in df.columns:
            df['OS_Browser'] = (
                df['OperatingSystems'].astype(str) + '_' + df['Browser'].astype(str)
            )

        if 'Region' in df.columns and 'TrafficType' in df.columns:
            df['Region_TrafficType'] = (
                df['Region'].astype(str) + '_' + df['TrafficType'].astype(str)
            )

    return df


def make_pipeline(X_data):
    numeric_features = X_data.select_dtypes(include=['number']).columns.tolist()
    categorical_features = X_data.select_dtypes(exclude=['number']).columns.tolist()

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ])

    model = XGBClassifier(
        n_estimators=1000,
        max_depth=5,
        learning_rate=0.035,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )

    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

In [3]:
results = []
models = {}

for level, name in [
    (0, '19A_Raw_Baseline'),
    (1, '19B_Core_Engineered'),
    (2, '19C_Interactions'),
    (3, '19D_Log_Transforms'),
    (4, '19E_Categorical_Interactions')
]:
    print('\n' + '=' * 70)
    print(name)
    print('=' * 70)

    X_train_fe = add_engineered_features(X_train, level)
    X_valid_fe = add_engineered_features(X_valid, level)

    print('Features:', X_train_fe.shape[1])

    pipeline = make_pipeline(X_train_fe)
    pipeline.fit(X_train_fe, y_train)

    predictions = pipeline.predict_proba(X_valid_fe)[:, 1]
    score = roc_auc_score(y_valid, predictions)

    results.append({
        'Experiment': name,
        'Features': X_train_fe.shape[1],
        'ROC_AUC': score
    })
    models[name] = pipeline

    print(f'ROC-AUC: {score:.6f}')


19A_Raw_Baseline
Features: 13
ROC-AUC: 0.941758

19B_Core_Engineered
Features: 13
ROC-AUC: 0.941758

19C_Interactions
Features: 13
ROC-AUC: 0.941758

19D_Log_Transforms
Features: 13
ROC-AUC: 0.941758

19E_Categorical_Interactions
Features: 13
ROC-AUC: 0.941758


In [4]:
results_df = pd.DataFrame(results).sort_values('ROC_AUC', ascending=False).reset_index(drop=True)

print('\n' + '=' * 70)
print('EXPERIMENT 19 RESULTS')
print('=' * 70)
print(results_df.to_string(index=False))

best_score = results_df.loc[0, 'ROC_AUC']
best_name = results_df.loc[0, 'Experiment']
previous_best = 0.941815
difference = best_score - previous_best

print('\nPrevious local best:', f'{previous_best:.6f}')
print('Best Experiment 19 model:', best_name)
print('Best Experiment 19 ROC-AUC:', f'{best_score:.6f}')
print('Difference vs previous best:', f'{difference:+.6f}')

if best_score > previous_best:
    print('\n🔥 NEW LOCAL BEST')
else:
    print('\nNo Experiment 19 model beat the current local best.')


EXPERIMENT 19 RESULTS
                  Experiment  Features  ROC_AUC
            19A_Raw_Baseline        13 0.941758
         19B_Core_Engineered        13 0.941758
            19C_Interactions        13 0.941758
          19D_Log_Transforms        13 0.941758
19E_Categorical_Interactions        13 0.941758

Previous local best: 0.941815
Best Experiment 19 model: 19A_Raw_Baseline
Best Experiment 19 ROC-AUC: 0.941758
Difference vs previous best: -0.000057

No Experiment 19 model beat the current local best.
